# ✅ MSTR 用户批量小写化执行 Notebook

本 Notebook 用于在 MicroStrategy / Strategy One 环境中，逐步筛选包含大写字母的用户，并由执行人输入编号确认后，实际将选中的用户字段小写化。

**执行原则**

- ✅ 先看清单，再选编号，最后执行。
- ✅ 不跳 cell，不重复执行修改 cell。
- ✅ 密码不写入 Notebook，不提交到 Git。
- ⚠️ 第 7 步会真实修改 MSTR 用户。

## 1. ✅ 导入依赖并加载工具函数

这个 cell 主要是准备函数，代码较长，默认折叠。正常情况下只需要运行一次。

如果这里报 `ModuleNotFoundError`，请先在当前 Python 环境执行：

```bash
pip install -r requirements.txt
```

In [ ]:
import math
import os
import re
from dataclasses import dataclass
from getpass import getpass
from typing import Iterable, Optional

import pandas as pd
from IPython.display import HTML, display
from mstrio.connection import Connection
from mstrio.users_and_groups.user import User, list_users

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

SYSTEM_USER_NAMES = {
    "administrator",
    "guest",
    "public / guest",
    "public/guest",
    "system administrator",
    "mstr user",
    "microstrategy administrator",
}

UPPERCASE_RE = re.compile(r"[A-Z]")
USER_DISPLAY_COLUMNS = ["display_name", "login_id", "trust_id", "enabled"]
PAGE_SIZE = 20

@dataclass(frozen=True)
class UserChange:
    id: str
    display_name: str
    login_id: str
    trust_id: str
    new_display_name: str
    new_login_id: str
    new_trust_id: str


def display_table(df: pd.DataFrame, columns: Optional[list[str]] = None, *, page: int = 1, page_size: int = PAGE_SIZE, title: str = "") -> None:
    """Display a paged table without the confusing default DataFrame index."""
    if columns:
        columns = [column for column in columns if column in df.columns]
        view = df[columns].copy()
    else:
        view = df.copy()

    total = len(view)
    if title:
        print(title)
    if total == 0:
        print("数量: 0")
        display(HTML("<em>没有数据。</em>"))
        return

    total_pages = max(1, math.ceil(total / page_size))
    page = max(1, min(page, total_pages))
    start = (page - 1) * page_size
    end = start + page_size
    page_view = view.iloc[start:end].fillna("")

    print(f"数量: {total} | 每页: {page_size} | 当前页: {page}/{total_pages}")
    if total_pages > 1:
        print("如需翻页，请修改本 cell 中的 page 变量后重新运行。")

    display(HTML(page_view.to_html(index=False, escape=False)))


def has_uppercase(value: object) -> bool:
    return isinstance(value, str) and bool(UPPERCASE_RE.search(value))


def lowercase_or_none(value: object) -> Optional[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    return str(value).lower()


def users_to_dataframe(users: list[User]) -> pd.DataFrame:
    rows = []
    for user in users:
        login_id = getattr(user, "username", None) or getattr(user, "abbreviation", None)
        rows.append(
            {
                "id": getattr(user, "id", None),
                "display_name": getattr(user, "name", None) or getattr(user, "full_name", None),
                "full_name": getattr(user, "full_name", None),
                "login_id": login_id,
                "trust_id": getattr(user, "trust_id", None),
                "enabled": getattr(user, "enabled", None),
            }
        )
    return pd.DataFrame(rows)


def find_uppercase_users(df: pd.DataFrame) -> pd.DataFrame:
    text_columns = ["display_name", "full_name", "login_id", "trust_id"]
    selected = df[text_columns]
    if hasattr(selected, "map"):
        uppercase_cells = selected.map(has_uppercase)
    else:
        uppercase_cells = selected.applymap(has_uppercase)
    return df[uppercase_cells.any(axis=1)].copy()


def filter_system_users(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    def is_system(row: pd.Series) -> bool:
        values = {
            str(row.get("display_name") or "").strip().lower(),
            str(row.get("full_name") or "").strip().lower(),
            str(row.get("login_id") or "").strip().lower(),
        }
        return bool(values & SYSTEM_USER_NAMES)

    system_mask = df.apply(is_system, axis=1)
    return df[~system_mask].copy(), df[system_mask].copy()


def build_changes(df: pd.DataFrame) -> list[UserChange]:
    changes = []
    for row in df.itertuples(index=False):
        display_name = row.display_name or row.full_name or row.login_id
        login_id = row.login_id
        trust_id = row.trust_id
        changes.append(
            UserChange(
                id=row.id,
                display_name=display_name or "",
                login_id=login_id or "",
                trust_id=trust_id or "",
                new_display_name=lowercase_or_none(display_name) or "",
                new_login_id=lowercase_or_none(login_id) or "",
                new_trust_id=lowercase_or_none(trust_id) or "",
            )
        )
    return changes


def validate_no_login_collisions(all_users_df: pd.DataFrame, changes: Iterable[UserChange]) -> None:
    changes = list(changes)
    changed_ids = {change.id for change in changes}
    existing_logins = {
        str(row.login_id).lower(): row.id
        for row in all_users_df.itertuples(index=False)
        if row.id not in changed_ids and pd.notna(row.login_id) and row.login_id
    }

    seen_in_batch: dict[str, str] = {}
    errors = []
    for change in changes:
        login = change.new_login_id
        if not login:
            errors.append(f"{change.display_name}: 登录 ID 为空，无法更新")
            continue
        if login in existing_logins:
            errors.append(f"{change.login_id} -> {login}: 与现有用户 ID {existing_logins[login]} 冲突")
        if login in seen_in_batch:
            errors.append(f"{change.login_id} -> {login}: 与本批次用户 ID {seen_in_batch[login]} 冲突")
        seen_in_batch[login] = change.id

    if errors:
        message = "发现小写化后的登录 ID 冲突，已停止执行：\n" + "\n".join(f"- {error}" for error in errors)
        raise RuntimeError(message)


def parse_selection(raw_value: str, valid_numbers: set[int]) -> list[int]:
    value = raw_value.strip().lower()
    if value in {"all", "全部"}:
        return sorted(valid_numbers)
    if not value:
        raise ValueError("输入不能为空。示例：1 或 1;3;5 或 all")

    parts = [part.strip() for part in value.split(";")]
    if any(not part for part in parts):
        raise ValueError("编号之间请用英文分号分隔，且不要留空。示例：1;3;5")
    if any(not part.isdigit() for part in parts):
        raise ValueError("只能输入数字编号、英文分号，或者 all。示例：1 或 1;3;5 或 all")

    selected = [int(part) for part in parts]
    invalid = [number for number in selected if number not in valid_numbers]
    if invalid:
        raise ValueError(f"以下编号不存在：{invalid}。请从表格中的 batch_no 选择。")

    return sorted(set(selected))


def ask_user_selection(valid_numbers: set[int]) -> list[int]:
    sample = "输入示例：单个编号 1；多个编号 1;3;5；全部 all"
    while True:
        raw_value = input(f"请输入要实际小写化的用户编号，或输入 all 选择全部。{sample}\n你的输入: ")
        try:
            return parse_selection(raw_value, valid_numbers)
        except ValueError as exc:
            print(f"输入无效：{exc}")
            print(sample)


def apply_changes(connection: Connection, changes: list[UserChange]) -> pd.DataFrame:
    results = []
    for change in changes:
        print(f"正在处理: {change.display_name} / {change.login_id} -> {change.new_login_id}")
        try:
            user = User(connection=connection, id=change.id)
            user.alter(
                username=change.new_login_id,
                full_name=change.new_display_name,
                trust_id=change.new_trust_id or None,
                journal_comment="Batch lowercase user display name, login ID and trusted auth ID.",
            )
            results.append(
                {
                    "display_name": change.display_name,
                    "login_id": change.login_id,
                    "new_display_name": change.new_display_name,
                    "new_login_id": change.new_login_id,
                    "new_trust_id": change.new_trust_id,
                    "success": True,
                    "message": "updated",
                }
            )
        except Exception as exc:
            results.append(
                {
                    "display_name": change.display_name,
                    "login_id": change.login_id,
                    "new_display_name": change.new_display_name,
                    "new_login_id": change.new_login_id,
                    "new_trust_id": change.new_trust_id,
                    "success": False,
                    "message": str(exc),
                }
            )
    return pd.DataFrame(results)



def close_connection_safely(connection: Connection) -> None:
    """Best-effort close/logout for different mstrio-py versions."""
    close_methods = ["close", "logout", "disconnect"]
    for method_name in close_methods:
        method = getattr(connection, method_name, None)
        if callable(method):
            method()
            print(f"✅ 已调用 connection.{method_name}() 关闭 MSTR 会话。")
            return
    print("⚠️ 当前 Connection 对象未发现 close/logout/disconnect 方法；请关闭 Notebook kernel，并依赖 MSTR 服务端 session timeout 清理。")

print(f"pandas version: {pd.__version__}")
print("✅ 工具函数加载完成。")

## 2. 🔐 配置并登录 MSTR Library

- 默认读取环境变量 `MSTR_LIBRARY_URL`、`MSTR_USERNAME`、`MSTR_PASSWORD`、`MSTR_LOGIN_MODE`。
- 如果未设置 `MSTR_PASSWORD`，会在运行 cell 时安全输入。
- 不要把真实密码写入 Notebook。

In [ ]:
library_url = os.getenv("MSTR_LIBRARY_URL", "https://env-363414.customer.cloud.microstrategy.com/MicroStrategyLibrary/")
username = os.getenv("MSTR_USERNAME", "mstr")
password = os.getenv("MSTR_PASSWORD") or getpass("请输入 MSTR 登录密码: ")
login_mode = int(os.getenv("MSTR_LOGIN_MODE", "1"))

connection = Connection(
    base_url=library_url,
    username=username,
    password=password,
    login_mode=login_mode,
)

print(f"✅ 已连接: {library_url}")

## 3. 📋 列出当前平台全部用户

这一页用于确认连接环境和用户总量。为了避免一次显示太多用户，默认每页显示 20 条。

- 修改 `all_users_page` 可以翻页。
- `login_id` 即 account login，因此不再重复显示 `account_login` 列。

In [ ]:
all_users_page = 1

users = list_users(connection=connection)
users_df = users_to_dataframe(users)

display_table(
    users_df,
    USER_DISPLAY_COLUMNS,
    page=all_users_page,
    page_size=PAGE_SIZE,
    title="当前平台全部用户",
)

## 4. 🔎 筛选当前包含大写字母的用户

筛选字段包括：

- `display_name`
- `full_name`
- `login_id`
- `trust_id`

默认每页显示 20 条。修改 `uppercase_users_page` 可以翻页。

In [ ]:
uppercase_users_page = 1

uppercase_df = find_uppercase_users(users_df)

display_table(
    uppercase_df,
    USER_DISPLAY_COLUMNS,
    page=uppercase_users_page,
    page_size=PAGE_SIZE,
    title="当前包含大写字母的用户",
)

## 5. ✅ 选择本次要实际小写化的用户

本步骤会默认剔除疑似平台自带用户，然后给候选用户生成 `batch_no`。

请输入你要处理的编号：

- `1`：只处理编号 1。
- `1;3;5`：处理编号 1、3、5。
- `all`：处理全部候选用户。

⚠️ 表格已经隐藏 DataFrame 默认索引，最前面的 `batch_no` 就是唯一需要输入的编号。

In [ ]:
candidate_page = 1
exclude_system_users = True

candidate_df, system_df = filter_system_users(uppercase_df)

if not system_df.empty:
    display_table(
        system_df,
        USER_DISPLAY_COLUMNS,
        page=1,
        page_size=PAGE_SIZE,
        title="疑似平台自带用户，默认剔除",
    )
else:
    print("✅ 未发现疑似平台自带用户。")

selectable_df = candidate_df.copy() if exclude_system_users else uppercase_df.copy()
selectable_df = selectable_df.reset_index(drop=True)
selectable_df.insert(0, "batch_no", range(1, len(selectable_df) + 1))

if selectable_df.empty:
    raise RuntimeError("没有可处理用户。请停止执行后续 cell。")

display_table(
    selectable_df,
    ["batch_no", *USER_DISPLAY_COLUMNS],
    page=candidate_page,
    page_size=PAGE_SIZE,
    title="可选择的小写化候选用户",
)

valid_numbers = set(selectable_df["batch_no"].tolist())
selected_numbers = ask_user_selection(valid_numbers)

target_df = selectable_df[selectable_df["batch_no"].isin(selected_numbers)].drop(columns=["batch_no"]).copy()
selected_view_df = selectable_df[selectable_df["batch_no"].isin(selected_numbers)]

print(f"✅ 本次确认要实际小写化的用户数量: {len(target_df)}")
display_table(selected_view_df, ["batch_no", *USER_DISPLAY_COLUMNS], page=1, page_size=len(selected_view_df) or PAGE_SIZE)

## 6. 🧾 生成最终变更清单并检查冲突

这一步不会修改 MSTR，只负责展示“旧值 -> 新值”并检查小写后的 `login_id` 是否冲突。

✅ 如果没有报错，才继续执行第 7 步。

In [ ]:
changes = build_changes(target_df)
validate_no_login_collisions(users_df, changes)

preview_df = pd.DataFrame([change.__dict__ for change in changes])

print(f"✅ 最终待执行用户数量: {len(preview_df)}")
display_table(
    preview_df,
    ["display_name", "login_id", "trust_id", "new_display_name", "new_login_id", "new_trust_id"],
    page=1,
    page_size=len(preview_df) or PAGE_SIZE,
)

## 7. 🚀 实际执行小写化

⚠️ 执行这个 cell 会真实修改 MSTR 用户。请只在第 6 步变更清单确认无误后执行。

In [ ]:
result_df = apply_changes(connection, changes)

print("执行结果：")
display_table(result_df, page=1, page_size=len(result_df) or PAGE_SIZE)

if not result_df["success"].all():
    raise RuntimeError("存在执行失败项，请查看 result_df 的 message 字段。")

print("✅ 本批次修改请求已执行完成。")

## 8. ✅ 复查当前全部大写用户

本步骤不会再列出全部平台用户，而是只列出“当前仍包含大写字母的用户”。

成功标准：当前大写用户列表中不再包含第 5 步选择的目标用户。

In [ ]:
uppercase_after_page = 1

refreshed_users = list_users(connection=connection)
refreshed_df = users_to_dataframe(refreshed_users)
uppercase_after_df = find_uppercase_users(refreshed_df)

print("当前仍包含大写字母的用户：")
display_table(
    uppercase_after_df,
    USER_DISPLAY_COLUMNS,
    page=uppercase_after_page,
    page_size=PAGE_SIZE,
)

processed_ids = {change.id for change in changes}
remaining_processed = uppercase_after_df[uppercase_after_df["id"].isin(processed_ids)]

print(f"本批次目标用户中仍包含大写字母的数量: {len(remaining_processed)}")
display_table(remaining_processed, USER_DISPLAY_COLUMNS, page=1, page_size=len(remaining_processed) or PAGE_SIZE)

if remaining_processed.empty:
    print("✅ 检查成功：当前大写用户列表中已不再包含本批次目标用户。")
    print("🎉 恭喜，批处理脚本运行成功！")
else:
    raise RuntimeError("检查失败：部分本批次目标用户仍包含大写字母。")

## 9. 🔚 关闭 MSTR 连接

运行完成并复查成功后，请执行本 cell 显式释放 MSTR session。

如果当前 `mstrio-py` 版本没有暴露关闭方法，请关闭 Notebook kernel，让服务端 session timeout 自动清理。


In [ ]:
close_connection_safely(connection)
